# Healthcare Analytics, End-Term Project

**BIA-810 Final Project | May 12, 2025**

**Topic:** Analyzing Market Dynamics & Revitalizing Brand Strategy for Market Cannibalization of an Injectable Anesthesia Drug

**Data:** Medicare CCLF Claims (2016-2018), HCP Demographics, Patient Demographics, Zip-to-Territory Mapping, ICD-10 Diagnosis Specialty Mapping

---

## Executive Summary

The cannibalization play has failed. Product 2 (J2250, midazolam) was launched to absorb the dropping share of Product 1 (J1885, ketorolac). Instead, Product 3 (J3010, fentanyl), a competitor, has captured almost all of that share and has now overtaken Product 1 in revenue terms. Product 2 has been quietly compressed into clinical irrelevance.

**Three findings that explain everything:**

1. **Wrong substitute.** Midazolam (J2250) is a benzodiazepine sedative; ketorolac (J1885) is an NSAID for pain. They are not in the same drug class and were never clinically substitutable. The cannibalization assumption was a portfolio-level error.
2. **Retention crisis, not acquisition crisis.** J2250 loses more than half its writers every year (46.4% retention) and has zero High-Volume prescribers. No level of new-writer recruitment can outrun this churn.
3. **Direct competitive displacement.** 58% of J2250 patients have also received J3010. The competitor is not just winning new patients, it is treating *our* patients.

**Top three sized actions:**

1. **Fix retention first.** Re-engage the lapsed J2250 writers from 2018 (~148 NPIs). Lift retention 46% → 65% in 12 months → +120 to +200 incremental claims.
2. **Convert Trialists into Rising Stars.** 90% of J2250 writers sit at 1-4 claims/year. Doubling claim volume for just 20% of the 284 Trialists adds ~+170 claims with zero acquisition cost.
3. **Declare 4 emergency territories.** St. Louis, Phoenix, LA-San Diego, New York. Together they lost 70+ J2250 claims while J3010 gained 165+ in the same markets in a single year.

---

## Methodology & Assumptions

| Decision | Choice / Value | Rationale |
| --- | --- | --- |
| Time window | 2016-2018 | Per project brief. 17 line items from 2015 dropped (out-of-window). |
| Brand attribution | Line-level `clm_line_hcpcs_cd` ∈ market basket | Each line item identifies a specific drug administration; aggregating to claim-ID would risk double-counting administrations of multiple basket products on the same encounter. Yields **15,262 brand line items** within **15,139 unique claim IDs** out of 28,368 total dataset rows. |
| Date field | `clm_from_dt` parsed via `pd.to_datetime`; `clm_thru_dt` normalized from mixed ISO/US formats | Consistent with project guidance. |
| ZIP code handling | Cast to 5-digit zero-padded string before territory join | Preserves leading zeros (e.g., 01104). |
| Diagnosis specialty | Initial alphabet of `clm_dgns_cd` mapped via Diagnosis Code Mapping | Per assignment hints. |
| HCP definition | Unique `fac_prvdr_npi_num` (10-digit NPI) | Standard claims convention. |
| New writer | NPI's first observed claim year for that brand | Subject to a 2016 boundary effect (data start year), flagged on Chart 12. |
| Continuing writer | NPI active in both years of the period | Per assignment. |
| Revenue / spend metric | `clm_line_cvrd_pd_amt` (Medicare-paid amount) | Cleanest available proxy for brand revenue. Medicare-only, not commercial book. |

---

## Strategic Reframe, Why Cannibalization Was Always Going to Fail

The case has been framed as a market-execution problem. The data shows it was structurally a **pharmacological** problem.

| HCPCS | Generic | Drug Class | Clinical Role |
| --- | --- | --- | --- |
| J1885 | Ketorolac tromethamine | NSAID | Short-term moderate-to-severe **pain**; pre/post-op analgesic |
| **J2250** | **Midazolam HCl** | **Benzodiazepine** | **Induction of general anesthesia / sedation** |
| J3010 | Fentanyl citrate | Synthetic opioid | Analgesia during anesthesia, induction, recovery |
| J2704 | Propofol | IV anesthetic / sedative | Sedation / induction |

**Insight:** J2250 (sedative) and J1885 (NSAID) are not clinically substitutable. They sit at different points in the perioperative pathway. The "cannibalization plan" assumed they were portfolio substitutes when in fact J2250's natural competitive set is **J2704 (propofol)**, both are induction/sedation agents, and, in mixed sedation-analgesia protocols, **J3010 (fentanyl)**.

**Strategic implication:** The brand strategy must be reset against the *real* competitive set. Detail aids, KOL programs, and clinical evidence generation should reposition J2250 in the induction / sedation conversation against J2704 and J3010, not against J1885. This single reframe is the most important strategic move available.


## Setup, Imports, Configuration, and Data Load


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# ---- Constants ----
PRODUCTS = ['J1885', 'J2250', 'J2704', 'J3010']
YEARS    = [2016, 2017, 2018]

PROD_LABELS = {
    'J1885': 'Product 1, Leader (J1885)',
    'J2250': 'Product 2, Variant (J2250)',
    'J2704': 'Product 4, Alt. Competitor (J2704)',
    'J3010': 'Product 3, Main Competitor (J3010)',
}
PROD_SHORT = {
    'J1885': 'Prod 1 (Leader)',
    'J2250': 'Prod 2 (Variant)',
    'J2704': 'Prod 4 (Alt. Comp.)',
    'J3010': 'Prod 3 (Main Comp.)',
}
P_COLORS  = {'J1885': '#1a3a5c', 'J2250': '#2e86c1', 'J2704': '#aed6f1', 'J3010': '#7fb3d3'}
YR_COLORS = ['#1a3a5c', '#2e86c1', '#7fb3d3']
B1, B2, B3 = '#1a3a5c', '#1f618d', '#2e86c1'
B4, B5, B6 = '#7fb3d3', '#aed6f1', '#d6eaf8'

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
})

# ---- Load + standardize ----
# Place the dataset in the same folder as this notebook before running.
import os
DATA_FILENAME = 'Healthcare analytics final merged dataset.xlsx'
DATA_PATH = DATA_FILENAME if os.path.exists(DATA_FILENAME) else os.path.join(os.environ.get('DATA_DIR', '.'), DATA_FILENAME)
df = pd.read_excel(DATA_PATH)

# Date normalization (handles mixed ISO and US formats in clm_thru_dt)
df['clm_from_dt'] = pd.to_datetime(df['clm_from_dt'], errors='coerce')
df['clm_thru_dt'] = pd.to_datetime(df['clm_thru_dt'], errors='coerce')
df['claim_quarter'] = df['clm_from_dt'].dt.to_period('Q').astype(str)

# Sanity-check assertions
assert len(df) == 28368, f"Expected 28,368 rows, got {len(df):,}"
assert (df['claim_year'] == 2015).sum() == 17, "Expected 17 records from 2015"

market_df = df[df['clm_line_hcpcs_cd'].isin(PRODUCTS) & df['claim_year'].isin(YEARS)].copy()
print(f"Total dataset rows: {len(df):,}")
print(f"Records excluded (2015 out-of-window): {(df['claim_year'] == 2015).sum()}")
print(f"Brand line-items in scope: {len(market_df):,}")
print(f"Unique claim IDs: {market_df['cur_clm_uniq_id'].nunique():,}")
print(f"Years: {sorted(market_df['claim_year'].unique())}")

# Core pivots used throughout
claims   = market_df.groupby(['claim_year', 'clm_line_hcpcs_cd'])['cur_clm_uniq_id'].count().unstack(fill_value=0)[PRODUCTS]
patients = market_df.groupby(['claim_year', 'clm_line_hcpcs_cd'])['bene_mbi_id'].nunique().unstack(fill_value=0)[PRODUCTS]
hcps     = market_df.groupby(['claim_year', 'clm_line_hcpcs_cd'])['fac_prvdr_npi_num'].nunique().unstack(fill_value=0)[PRODUCTS]
revenue  = market_df.groupby(['claim_year', 'clm_line_hcpcs_cd'])['clm_line_cvrd_pd_amt'].sum().unstack(fill_value=0)[PRODUCTS]

## Chart Helper Functions


In [ ]:
def stacked_100_chart(pivot_df, title, ylabel, footnote):
    pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('white')
    x = np.arange(len(YEARS))
    bottoms = np.zeros(len(YEARS))
    for prod in PRODUCTS:
        vals = pct[prod].values
        ax.bar(x, vals, bottom=bottoms, color=P_COLORS[prod], label=PROD_LABELS[prod], width=0.5, zorder=3)
        for j, (v, b) in enumerate(zip(vals, bottoms)):
            if v >= 7:
                ax.text(j, b + v / 2, f"{v:.1f}%", ha='center', va='center',
                        color='white', fontsize=11, fontweight='bold', zorder=4)
            elif v >= 3:
                ax.text(j, b + v / 2, f"{v:.1f}%", ha='center', va='center',
                        color='#1a3a5c', fontsize=9, fontweight='bold', zorder=4)
        bottoms += vals
    for y in [20, 40, 60, 80, 100]:
        ax.axhline(y, color='#dddddd', linewidth=0.7, zorder=1)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=13)
    ax.set_xlabel('Calendar Year', fontsize=13, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=14)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14),
              ncol=2, fontsize=9.5, frameon=True, edgecolor='#cccccc', framealpha=0.95)
    fig.text(0.5, 0.01, footnote, ha='center', fontsize=8, color='#888888', style='italic')
    plt.subplots_adjust(left=0.09, right=0.97, top=0.88, bottom=0.22)
    plt.show()

def line_chart_2x2(data_df, title, ylabel, footnote):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.patch.set_facecolor('white')
    axes = axes.flatten()
    x = np.arange(len(YEARS))
    for i, prod in enumerate(PRODUCTS):
        ax = axes[i]
        vals = data_df[prod].values
        ax.plot(x, vals, color=P_COLORS[prod], marker='o', linewidth=2.5, markersize=9, zorder=4)
        for j, v in enumerate(vals):
            ax.annotate(f"{v:.2f}", xy=(j, v), xytext=(0, 12), textcoords='offset points',
                        ha='center', va='bottom', fontsize=10, fontweight='bold', color=P_COLORS[prod],
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9), zorder=5)
        ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=11)
        ax.set_ylim(0, max(data_df.values.max() * 1.35, 1))
        ax.set_title(PROD_LABELS[prod], fontsize=10, fontweight='bold', color=P_COLORS[prod], pad=8)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.0)
    fig.text(0.5, 0.01, footnote, ha='center', fontsize=8, color='#888888', style='italic')
    fig.text(0.04, 0.5, ylabel, va='center', rotation='vertical', fontsize=12, color='#333333')
    plt.subplots_adjust(left=0.1, right=0.97, top=0.90, bottom=0.1, hspace=0.45, wspace=0.3)
    plt.show()

---

# Q1, Market Dynamics & Competitive Landscape Assessment

Comprehensive analysis of the injectable anesthesia market for 2016-2018: trends in market share, growth rates, and key indicators for Products 1, 2, 3, and 4. We examine claims, patients, and HCPs (writers); claims-per-HCP and patients-per-HCP productivity; and the territory-level dynamics of the variant brand's decline against the main competitor's rise.


### Chart 1, Claims Share by Product (100% Stacked Bar)


In [ ]:
stacked_100_chart(
    claims,
    'Injectable Anesthesia Market, Claims Share by Product\n2016, 2018',
    'Share of Total Claims (%)',
    'Market Basket: J1885, J2250, J2704, J3010  |  Source: Medicare CCLF Claims'
)
print(claims.to_string())

**What it shows:** Product 1 (Leader) held 73.8% of all claims in 2016 but dropped to 58.9% by 2018. Product 3 (Main Competitor) grew from 14.3% to 28.8%. Product 2 (our Variant) barely moved, from 10.0% to just 5.4%, meaning it lost share rather than absorbing Product 1's drop.

**Insight:** The cannibalization strategy has failed completely. Product 2 was launched to absorb Product 1's dropping share. Instead, Product 3 is taking it. Every percentage point that Product 1 loses is going to the competitor, not our own brand.

**Recommendation:** Product 2 needs an urgent commercial reset. The go-to-market strategy, messaging, targeting, and channel mix, must be redesigned before Product 1 loses more ground and Product 3 becomes the default alternative in the market.


### Chart 2, Unique Patient Share by Product


In [ ]:
stacked_100_chart(
    patients,
    'Injectable Anesthesia Market, Unique Patient Share by Product\n2016, 2018',
    'Share of Unique Patients (%)',
    'Market Basket: J1885, J2250, J2704, J3010  |  Source: Medicare CCLF Claims'
)
print(patients.to_string())

**What it shows:** Product 1's patient share fell from 66.4% to 51.3%. Product 3's patient share grew from 17.9% to 32.2%. Product 2's patient share fell from 13.0% to 7.2%.

**Insight:** Product 3 is not just winning new patients, it is treating patients who previously received Product 1 and Product 2. The patient base is actively shifting to the competitor. At the current rate, Product 3 will surpass Product 1 in patient share within 1-2 years.

**Recommendation:** Patient retention programs must be launched immediately. HCPs who currently treat patients with Product 2 should receive personalized outreach focused on the clinical benefits of staying on the brand. Switching patients back once they have moved to the competitor is far harder than retaining them now.


### Chart 3, Unique HCP (Writer) Share by Product


In [ ]:
stacked_100_chart(
    hcps,
    'Injectable Anesthesia Market, Unique HCP (Writer) Share by Product\n2016, 2018',
    'Share of Unique HCPs (%)',
    'HCP = unique NPI ID  |  Market Basket: J1885, J2250, J2704, J3010  |  Source: Medicare CCLF Claims'
)
print(hcps.to_string())

**What it shows:** Product 2's HCP writer share fell from 23.9% to 15.5%. Product 3's writer share grew from 28.8% to 32.1%. Product 1 dropped from 41.2% to 34.0%.

**Insight:** Product 2 is losing HCPs faster than it is losing patients. This is a leading indicator, when doctors stop writing a brand, patients follow shortly after. The HCP writer base is the foundation of any brand's commercial performance, and Product 2's foundation is collapsing.

**Recommendation:** Immediately identify and prioritize the HCPs who switched away from Product 2 in 2018. Deploy the sales force to re-engage these specific NPIs with a targeted clinical message. For Disease Aware writers, those who wrote only once, use targeted samples and case studies to push them into the Trialist tier.


### Chart 4, Claims per HCP by Product


In [ ]:
claims_per_hcp = (claims / hcps).round(2)
line_chart_2x2(
    claims_per_hcp,
    'Claims per HCP (Writer) by Product\n2016, 2018',
    'Claims per Unique HCP',
    'Claims per HCP = Total Claims / Unique NPI IDs per year  |  Source: Medicare CCLF Claims'
)
print(claims_per_hcp.to_string())

**What it shows:** Product 1 writers average 6.41-7.46 claims per HCP. Product 3 writers grew from 1.77 to 3.45 claims per HCP. Product 2 writers average only 1.34-1.51, barely above one claim per writer per year.

**Insight:** Product 2 writers are not committed. They write the brand once or twice and stop. Product 3 writers are becoming increasingly productive, each one is writing more claims every year, which means deeper clinical adoption. Product 2 has the opposite trajectory.

**Recommendation:** Focus sales effort on converting existing Product 2 Trialist writers (2-4 claims) into Rising Star writers (5-9 claims). A rep-delivered clinical case study program or peer speaker event featuring a high-volume Product 2 writer is the highest-leverage move available.


### Chart 5, Patients per HCP by Product


In [ ]:
patients_per_hcp = (patients / hcps).round(2)
line_chart_2x2(
    patients_per_hcp,
    'Patients per HCP (Writer) by Product\n2016, 2018',
    'Patients per Unique HCP',
    'Patients per HCP = Unique Patients / Unique NPI IDs per year  |  Source: Medicare CCLF Claims'
)
print(patients_per_hcp.to_string())

**What it shows:** Product 3 writers treated 2.7 patients each in 2018, up from 1.58 in 2016. Product 2 writers treated only 1.22 patients each, and this number is declining. Product 1 writers treat 4.1 patients each and hold steady.

**Insight:** Product 3 HCPs are building patient panels around the brand. Product 2 HCPs are not. When a doctor regularly treats multiple patients with the same product, it becomes embedded in their clinical routine, making them far less likely to switch.

**Recommendation:** Identify Product 2 writers who treat 2 or more patients and treat them as high-priority accounts. These writers are building a modest patient panel, with the right support (patient assistance programs, adherence tools, follow-up materials), they can be converted into long-term loyalists.


### Charts 6 & 7, Top 5 Declining Territories for J2250 vs J3010


In [ ]:
# Compute top-5 declining territories for J2250 (YoY 2017→2018)
terr = (market_df
        .groupby(['Territory Name', 'claim_year', 'clm_line_hcpcs_cd'])['cur_clm_uniq_id']
        .count().reset_index())
terr.columns = ['territory', 'year', 'product', 'claims']

j2250_terr = terr[terr['product'] == 'J2250'].pivot_table(
    index='territory', columns='year', values='claims', fill_value=0)
j2250_terr.columns = [int(c) for c in j2250_terr.columns]
for y in YEARS:
    if y not in j2250_terr.columns: j2250_terr[y] = 0
j2250_terr = j2250_terr[YEARS]
j2250_terr['yoy_1718'] = ((j2250_terr[2018] - j2250_terr[2017]) /
                          j2250_terr[2017].replace(0, np.nan)) * 100
top5 = j2250_terr.sort_values('yoy_1718').head(5)
top5_names = top5.index.tolist()
yoy_dict   = top5['yoy_1718'].to_dict()

j3010_terr = (terr[(terr['product'] == 'J3010') & (terr['territory'].isin(top5_names))]
              .pivot_table(index='territory', columns='year', values='claims', fill_value=0))
j3010_terr.columns = [int(c) for c in j3010_terr.columns]
for y in YEARS:
    if y not in j3010_terr.columns: j3010_terr[y] = 0
j3010_terr = j3010_terr[YEARS].reindex(top5_names, fill_value=0)

SHORT = {
    'St Louis, MO':     'St. Louis,\nMO',
    'Phoenix, AZ':      'Phoenix,\nAZ',
    'LA-San Diego, CA': 'LA-San\nDiego, CA',
    'New York, NY':     'New York,\nNY',
    'Minneapolis, MN':  'Minneapolis,\nMN',
}

def clustered_bar(data_df, title, ylabel, footnote, show_yoy=False):
    bw = 0.22
    x = np.arange(len(top5_names))
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('white')
    for i, yr in enumerate(YEARS):
        offsets = x + (i - len(YEARS) / 2 + 0.5) * bw
        vals    = [data_df.loc[t, yr] for t in top5_names]
        ax.bar(offsets, vals, width=bw, color=YR_COLORS[i], label=str(yr), zorder=3)
        for j, v in enumerate(vals):
            if v > 0:
                ax.text(offsets[j], v + 0.8, str(int(v)),
                        ha='center', va='bottom', fontsize=9, fontweight='bold', color=YR_COLORS[i])
    if show_yoy:
        for j, t in enumerate(top5_names):
            yoy = yoy_dict[t]
            col = '#c0392b' if yoy < 0 else '#1e8449'
            ax.text(j, -data_df.values.max() * 0.16,
                    f"YoY '17→'18:\n{yoy:+.1f}%",
                    ha='center', va='top', fontsize=8.5, fontweight='bold', color=col)
    ax.set_xticks(x); ax.set_xticklabels([SHORT.get(t, t) for t in top5_names], fontsize=11)
    ax.set_xlabel('Territory', fontsize=13, labelpad=12)
    ax.set_ylabel(ylabel, fontsize=12, labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=16)
    ymax = data_df.values.max()
    ax.set_ylim(-ymax * 0.28 if show_yoy else 0, ymax * 1.3)
    ax.legend(title='Year', loc='upper right', fontsize=10)
    fig.text(0.5, 0.01, footnote, ha='center', fontsize=8, color='#888888', style='italic')
    plt.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.26 if show_yoy else 0.16)
    plt.show()

# Chart 6, J2250 in top-5 declining territories
clustered_bar(
    top5[YEARS],
    'Product 2 (Variant, J2250): Claims in Top 5 Territories\nWith Largest Drop in Volume (2017 → 2018)',
    'Number of Claims',
    'Territories ranked by largest % decline in J2250 claims from 2017 to 2018  |  Source: Medicare CCLF Claims',
    show_yoy=True
)
print("Top 5 Declining Territories, J2250:")
print(top5[YEARS + ['yoy_1718']].round(1).to_string())

# Chart 7, J3010 in same 5 territories
clustered_bar(
    j3010_terr,
    'Product 3 (Main Competitor, J3010): Claims in Same 5 Territories\n2016, 2018',
    'Number of Claims',
    'Same 5 territories as Chart 6, competitor gaining where Variant brand declines  |  Source: Medicare CCLF Claims',
    show_yoy=False
)
print("\nSame 5 Territories, J3010:")
print(j3010_terr.to_string())

**What it shows:** Five territories had the steepest Product 2 declines from 2017 to 2018, St. Louis (-72.7%), Phoenix (-70.0%), LA-San Diego (-57.9%), New York (-57.5%), and Minneapolis (-53.3%). All five dropped by more than half in a single year. In every one of these five territories, Product 3 grew. New York saw Product 3 grow from 33 to 153 claims, a 364% increase, while Product 2 dropped from 40 to 17.

**Insight:** This is not coincidence. Product 3 is directly capturing the share that Product 2 is vacating. The competitor is executing more effectively in exactly the markets where our brand is weakest, a 50-70% drop across five different geographies in one year signals a systemic failure.

**Recommendation:** Audit rep activity logs and formulary status in all five territories. For large markets like New York and LA-San Diego, increase rep call frequency immediately. For smaller markets like St. Louis and Phoenix, deploy targeted NPP campaigns and KOL programs. Use these 5 territories as competitive intelligence battlegrounds, understand what Product 3 is doing differently and replicate.


### Chart 7-Mirror, Direct Competitive Displacement (NEW)

A side-by-side mirror chart that puts J2250 losses and J3010 gains on the same axis to show the one-for-one displacement in absolute claim counts.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')
x = np.arange(len(top5_names))
bw = 0.35
j2250_change = [int(top5.loc[t, 2018] - top5.loc[t, 2017]) for t in top5_names]
j3010_change = [int(j3010_terr.loc[t, 2018] - j3010_terr.loc[t, 2017]) for t in top5_names]

ax.bar(x - bw/2, j2250_change, width=bw, color='#c0392b', label='J2250 Net Change (2017→2018)', zorder=3)
ax.bar(x + bw/2, j3010_change, width=bw, color='#1e8449', label='J3010 Net Change (2017→2018)', zorder=3)
for i, (a, b) in enumerate(zip(j2250_change, j3010_change)):
    ax.text(i - bw/2, a - 2 if a < 0 else a + 1, f"{a:+d}", ha='center',
            va='top' if a < 0 else 'bottom', fontsize=10, fontweight='bold', color='#c0392b')
    ax.text(i + bw/2, b + 1, f"{b:+d}", ha='center', va='bottom',
            fontsize=10, fontweight='bold', color='#1e8449')

ax.axhline(0, color='#333333', linewidth=1.2)
ax.set_xticks(x); ax.set_xticklabels([SHORT.get(t, t) for t in top5_names], fontsize=11)
ax.set_xlabel('Territory', fontsize=13, labelpad=12)
ax.set_ylabel('Net Change in Claims (Count)', fontsize=12, labelpad=10)
ax.set_title('Direct Competitive Displacement, Net Claim Movement (2017 → 2018)\nIn the 5 Worst-Declining Territories for J2250',
             fontsize=13, fontweight='bold', pad=14)
ax.legend(loc='upper left', fontsize=10)
fig.text(0.5, 0.01,
         'Negative bars = J2250 losses  |  Positive bars = J3010 gains  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.16)
plt.show()

print(f"\nTotal J2250 net loss across 5 territories: {sum(j2250_change):+d} claims")
print(f"Total J3010 net gain across same 5 territories: {sum(j3010_change):+d} claims")

**Insight:** One-glance evidence of one-for-one displacement. Every territory where J2250 lost claims, J3010 gained more than the loss. The transfer is direct and asymmetric, the market is not shrinking; it is being captured.

**Recommendation:** This chart is the single strongest visual proof of competitive displacement. Use it as the opening visual in the strategic-recommendation slide of the panel presentation.


---

# Q2, Key Market Drivers of the Injectable Anesthesia Market

The Primary Market Research team has identified Patient Age, Diagnosis Specialty, HCP Specialty, and New Prescriber growth as the key Market Drivers. We investigate the trends in each and translate them into actionable levers for Product 2.


### Chart 8, Top 5 Diagnosis Specialties


In [ ]:
diag_counts = market_df['Diagnosis_Specialty'].value_counts()
top5_diag   = diag_counts.head(5)
other_count = diag_counts.iloc[5:].sum()
diag_plot   = pd.concat([top5_diag, pd.Series({'Other': other_count})])
total_claims_count = diag_plot.sum()
DIAG_COLORS = [B1, B2, B3, B4, B5, '#5dade2']
DIAG_ABBR = {
    'Circulatory System':                                                  'Circulatory System',
    'Factors Influencing Health Status and Contact with Health Services':  'Factors Influencing Health Status',
    'Symptoms, Signs and Abnormal Clinical and Lab Findings':              'Symptoms, Signs & Abnormal Findings',
    'Musculoskeletal and Connective Tissue':                               'Musculoskeletal & Connective Tissue',
    'Endocrine, Nutritional, Metabolic':                                   'Endocrine, Nutritional, Metabolic',
    'Other':                                                               'Other',
}

fig = plt.figure(figsize=(16, 6), facecolor='white')
gs  = GridSpec(1, 2, width_ratios=[1, 1.5], wspace=0.06)
ax_donut = fig.add_subplot(gs[0])
ax_tbl   = fig.add_subplot(gs[1])
pcts     = diag_plot / total_claims_count * 100
wedges, _ = ax_donut.pie(diag_plot.values, colors=DIAG_COLORS, startangle=90,
                          wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
for wedge, pct in zip(wedges, pcts.values):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x_t   = 0.75 * np.cos(np.radians(angle))
    y_t   = 0.75 * np.sin(np.radians(angle))
    ax_donut.text(x_t, y_t, f"{pct:.1f}%", ha='center', va='center', fontsize=10, fontweight='bold',
                  color='white' if pct > 8 else B1)
ax_donut.text(0, 0, f"Total\n{total_claims_count:,.0f}\nClaims",
              ha='center', va='center', fontsize=10, fontweight='bold', color=B1)
ax_donut.set_title('Top 5 Diagnosis Specialties\n(All Market Claims, 2016-2018)',
                   fontsize=13, fontweight='bold', pad=14)
ax_tbl.axis('off')
tbl_data = [[DIAG_ABBR.get(name, name), f"{int(cnt):,}", f"{cnt/total_claims_count*100:.1f}%"]
            for name, cnt in diag_plot.items()]
tbl = ax_tbl.table(cellText=tbl_data, colLabels=['Diagnosis Specialty', 'Claims', 'Share'],
                   loc='center', cellLoc='left', bbox=[0.0, 0.05, 1.0, 0.9])
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5)
for c in range(3):
    tbl[(0, c)].set_facecolor(B1); tbl[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, len(tbl_data) + 1):
    for c in range(3):
        tbl[(r, c)].set_edgecolor('#dddddd')
        tbl[(r, c)].set_facecolor(B6 if r % 2 == 0 else 'white')
        if c == 0: tbl[(r, c)].set_text_props(color=DIAG_COLORS[r-1], fontweight='bold')
ax_tbl.set_title('Diagnosis Specialty Detail', fontsize=11, fontweight='bold', pad=8, color=B1)
fig.text(0.5, 0.01, 'All four market basket products included  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.08)
plt.show()
print(diag_plot.to_string())

**What it shows:** Nearly half of all market claims, 48%, are linked to Circulatory System diagnoses. Factors Influencing Health Status (11.9%), Symptoms & Signs (8.4%), Musculoskeletal (7.8%), and Endocrine/Metabolic (5.0%) make up the next four.

**Insight:** This is a cardiovascular-dominant market. Any product that wins the trust of cardiologists and vascular surgeons wins the market. Product 3's rapid growth strongly suggests it has found a foothold in circulatory care that Product 2 has not.

**Recommendation:** All sales messaging and clinical detail aids for Product 2 must be anchored in cardiovascular outcomes. If the brand has no differentiated cardiovascular data, generating real-world evidence in this indication should be a medical affairs priority for the next 12 months.


### Chart 9, Unique HCP Writers by Specialty


In [ ]:
hcp_spec = market_df.groupby('HCP_Specialty')['fac_prvdr_npi_num'].nunique().sort_values(ascending=True)
SPEC_COLORS = [B5, B4, B3, B2, B1]

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('white')
y_pos = np.arange(len(hcp_spec))
ax.barh(y_pos, hcp_spec.values, color=SPEC_COLORS, height=0.55, zorder=3)
for i, v in enumerate(hcp_spec.values):
    ax.text(v + hcp_spec.values.max() * 0.015, i, f"{v:,}",
            ha='left', va='center', fontsize=11, fontweight='bold', color=SPEC_COLORS[i])
ax.set_yticks(y_pos); ax.set_yticklabels(hcp_spec.index, fontsize=12)
ax.set_xlim(0, hcp_spec.values.max() * 1.22)
ax.set_xlabel('Number of Unique HCPs (NPI IDs)', fontsize=12, labelpad=10)
ax.set_title('Number of Unique HCP Writers by Specialty\nAll Market Basket Products Combined  |  2016, 2018',
             fontsize=14, fontweight='bold', pad=16)
fig.text(0.5, 0.01, 'HCP = unique NPI ID  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.16, right=0.97, top=0.88, bottom=0.18)
plt.show()
print(hcp_spec.sort_values(ascending=False).to_string())

**What it shows:** Anesthesiologists are the dominant writer group with 228 unique HCPs, more than double any other specialty. Cardiologists (86), Orthopedics (64), Gastroenterology (63), and Neurology (58) are roughly equal contributors.

**Insight:** Anesthesiologists are the primary decision-makers in this market. Winning or losing an anesthesiology group at a major hospital has a disproportionate impact on total brand volume. The four secondary specialties each represent meaningful but different clinical conversations.

**Recommendation:** Protect the Anesthesiology base at all costs, this is a must-win segment. Simultaneously, design specialty-specific NPP content for each of the four secondary specialties. A cardiologist needs a different clinical message than an orthopedic surgeon.


### Charts 10 & 11, Patient Age Distribution


In [ ]:
def age_bucket(age):
    if age <= 30:   return '18-30'
    elif age <= 40: return '31-40'
    elif age <= 50: return '41-50'
    elif age <= 60: return '51-60'
    elif age <= 70: return '61-70'
    elif age <= 80: return '71-80'
    else:           return '81+'

market_df['age_bucket'] = market_df['Age'].apply(age_bucket)
AGE_ORDER  = ['18-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81+']
AGE_COLORS = [B1, B2, B3, B3, B4, B4, B5]

age_counts     = market_df.groupby('age_bucket')['bene_mbi_id'].nunique().reindex(AGE_ORDER, fill_value=0)
age_pcts       = age_counts / age_counts.sum() * 100
age_claim_pcts = market_df.groupby('age_bucket').size().reindex(AGE_ORDER, fill_value=0)
age_claim_pcts = age_claim_pcts / age_claim_pcts.sum() * 100

fig = plt.figure(figsize=(13, 9), facecolor='white')
gs  = GridSpec(2, 1, height_ratios=[1, 1], hspace=0.4)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
x = np.arange(len(AGE_ORDER)); bw = 0.55

# Chart 10, unique patients
ax1.bar(x, age_counts.values, width=bw, color=AGE_COLORS, zorder=3)
for i, v in enumerate(age_counts.values):
    ax1.text(i, v + age_counts.max() * 0.02, f"{v:,}",
             ha='center', va='bottom', fontsize=10, fontweight='bold', color=AGE_COLORS[i])
ax1.set_xticks(x); ax1.set_xticklabels(AGE_ORDER, fontsize=11)
ax1.set_ylabel('Unique Patients', fontsize=12, labelpad=10)
ax1.set_title('Injectable Anesthesia Market, Unique Patients by Age Group  |  2016, 2018',
              fontsize=13, fontweight='bold', pad=12)
ax1.set_ylim(0, age_counts.max() * 1.2)

# Chart 11, % of claims
ax2.bar(x, age_claim_pcts.values, width=bw, color=AGE_COLORS, zorder=3)
for i, v in enumerate(age_claim_pcts.values):
    ax2.text(i, v + age_claim_pcts.max() * 0.02, f"{v:.1f}%",
             ha='center', va='bottom', fontsize=10, fontweight='bold', color=AGE_COLORS[i])
ax2.set_xticks(x); ax2.set_xticklabels(AGE_ORDER, fontsize=11)
ax2.set_xlabel('Patient Age Group', fontsize=12, labelpad=10)
ax2.set_ylabel('Share of Total Claims (%)', fontsize=12, labelpad=10)
ax2.set_title('Injectable Anesthesia Market, % of Claims by Patient Age Group  |  2016, 2018',
              fontsize=13, fontweight='bold', pad=12)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_ylim(0, age_claim_pcts.max() * 1.2)

fig.text(0.5, 0.01, 'Market Basket: J1885, J2250, J2704, J3010  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.09, right=0.97, top=0.95, bottom=0.06)
plt.show()
print("Unique patients by age bucket:")
print(age_counts.to_string())
print("\n% of claims by age bucket:")
print(age_claim_pcts.round(1).to_string())

**What it shows:** The 61-70 age group has the most unique patients at 778 (21.2% of claims). Patients 61 and older account for more than half of all unique patients and 54.2% of total claims. The 18-30 group (499 patients, 13.0% of claims) is surprisingly large.

**Insight:** The core patient population is older adults, consistent with a market driven by surgical procedures in an aging population. Older patients also generate disproportionately more claims per person, meaning more procedures, follow-ups, and treatment interactions. This makes them the highest-value patient segment by far. The unexpectedly large 18-30 segment suggests injectable anesthesia is also being used in non-elective procedures (childbirth, trauma, dental).

**Recommendation:** Position Product 2 as the preferred option for complex, multi-procedure older patients, feature safety profile, drug-interaction data, and dosing flexibility for elderly patients prominently. For the 18-30 group, push digital-first patient education through HCP platforms.


### Chart 12, New HCP Writers per Product per Year

> **Data caveat (boundary effect):** 2016 "new writer" counts equal total writers because the data window starts in 2016. Read 2017 and 2018 as the true acquisition trend.


In [ ]:
new_writers = {}
for prod in PRODUCTS:
    prod_df  = market_df[market_df['clm_line_hcpcs_cd'] == prod]
    first_yr = prod_df.groupby('fac_prvdr_npi_num')['claim_year'].min()
    vc       = first_yr.value_counts().sort_index()
    new_writers[prod] = {yr: int(vc.get(yr, 0)) for yr in YEARS}

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('white')
bw = 0.18; x = np.arange(len(YEARS))
for i, prod in enumerate(PRODUCTS):
    vals    = [new_writers[prod][yr] for yr in YEARS]
    offsets = x + (i - len(PRODUCTS) / 2 + 0.5) * bw
    ax.bar(offsets, vals, width=bw, color=P_COLORS[prod], label=PROD_LABELS[prod], zorder=3)
    for j, v in enumerate(vals):
        if v > 0:
            ax.text(offsets[j], v + 3, str(v), ha='center', va='bottom',
                    fontsize=9, fontweight='bold', color=P_COLORS[prod])
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=13)
ax.set_xlabel('Calendar Year', fontsize=12, labelpad=10)
ax.set_ylabel('Number of New Writers (NPI IDs)', fontsize=11, labelpad=10)
ax.set_title('New HCP Writers per Product per Year\n2016, 2018', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='upper right', fontsize=9)
fig.text(0.5, 0.01,
         "BOUNDARY EFFECT: 2016 'new writer' counts equal total writers because data window starts in 2016  |  Source: Medicare CCLF Claims",
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.09, right=0.97, top=0.88, bottom=0.14)
plt.show()
print(pd.DataFrame(new_writers, index=YEARS).to_string())

**What it shows:** Product 2 acquired 285 new writers in 2016, 111 in 2017, and only 34 in 2018. Product 1 had 492 new writers in 2016 but virtually zero since, it is a fully mature brand. Product 4 acquired 152 new writers in 2017 and 140 in 2018, consistent acquisition.

**Insight:** Product 2's new writer pipeline has nearly collapsed. At 34 new writers in 2018, the brand is one year away from zero new acquisitions. This is a brand on the verge of clinical irrelevance. Product 4, the alternative competitor, is quietly growing its base.

**Recommendation:** New writer acquisition must become a dedicated commercial objective with its own KPI tracking and incentive structure, but only after retention is fixed (see Q3 priority 1). Acquiring writers we cannot retain is wasted spend.


### Chart 13, Continuing HCP Writers per Product


In [ ]:
cont_writers = {}
for prod in PRODUCTS:
    prod_df = market_df[market_df['clm_line_hcpcs_cd'] == prod]
    writers_by_year = {yr: set(prod_df[prod_df['claim_year'] == yr]['fac_prvdr_npi_num']) for yr in YEARS}
    cont_writers[prod] = {
        '2016→2017': len(writers_by_year[2016] & writers_by_year[2017]),
        '2017→2018': len(writers_by_year[2017] & writers_by_year[2018])
    }

PERIODS = ['2016→2017', '2017→2018']
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
x = np.arange(len(PERIODS))
markers = ['o', 's', '^', 'D']
for prod, mkr in zip(PRODUCTS, markers):
    vals = [cont_writers[prod][p] for p in PERIODS]
    ax.plot(x, vals, color=P_COLORS[prod], marker=mkr, linewidth=2.5, markersize=9, label=PROD_LABELS[prod])
    for j, v in enumerate(vals):
        ax.text(j, v + 12, str(v), ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=P_COLORS[prod])
ax.set_xticks(x); ax.set_xticklabels(PERIODS, fontsize=13)
ax.set_xlabel('Period', fontsize=12, labelpad=10)
ax.set_ylabel('Number of Continuing Writers (NPI IDs)', fontsize=11, labelpad=10)
ax.set_title('Continuing HCP Writers per Product\n(Writers Active in Both Years of Each Period)',
             fontsize=13, fontweight='bold', pad=14)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=9.5)
fig.text(0.5, 0.01, 'Continuing Writer = NPI active in both years of the period  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.1, right=0.97, top=0.88, bottom=0.30)
plt.show()
print(pd.DataFrame(cont_writers).T.to_string())

**What it shows:** Product 1 retains essentially 100% of its writers every year. Product 3's continuing writer base grew from 297 to 393, a 32% increase in retained writers. Product 2 lost continuing writers, from 165 down to 128, a 22% decline. Product 4 grew from 35 to 103.

**Insight:** Product 2 is losing writers from both ends simultaneously, fewer new writers coming in and more existing writers leaving. This double erosion means the total writer base will continue shrinking every year unless the retention crisis is addressed.

**Recommendation:** Before investing in new writer acquisition, fix retention. Pull the NPI list of the 37 writers who continued with Product 2 in 2016→2017 but left in 2017→2018. Personally contact each one through the most senior territory rep.


---

# Additional Strategic Analytics

These supplementary analyses go beyond the required charts to test the cannibalization hypothesis directly, surface revenue dynamics, and segment HCPs in ways that drive the Q3 recommendations.


### Chart A, Absolute Claims Volume + YoY Growth + CAGR


In [ ]:
yoy = claims.pct_change() * 100
cagr = ((claims.iloc[-1] / claims.iloc[0]) ** (1/2) - 1) * 100

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('white')
x = np.arange(len(YEARS))
markers = ['o', 's', '^', 'D']
for prod, mkr in zip(PRODUCTS, markers):
    vals = claims[prod].values
    ax.plot(x, vals, color=P_COLORS[prod], marker=mkr, linewidth=2.5, markersize=9, label=PROD_LABELS[prod])
    for j, v in enumerate(vals):
        ax.text(j, v + 80, f"{int(v):,}", ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=P_COLORS[prod])
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=13)
ax.set_xlabel('Calendar Year', fontsize=13, labelpad=10)
ax.set_ylabel('Total Claims (Count)', fontsize=12, labelpad=10)
ax.set_title('Injectable Anesthesia Market, Absolute Claims Volume by Product\n2016, 2018',
             fontsize=14, fontweight='bold', pad=16)
ax.set_ylim(0, claims.values.max() * 1.45)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=9.5)
fig.text(0.5, 0.01, 'Market Basket: J1885, J2250, J2704, J3010  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.09, right=0.97, top=0.88, bottom=0.28)
plt.show()

print("Year-over-Year Growth Rate (%):")
print(yoy.round(1).to_string())
print(f"\nCAGR 2016 → 2018:")
print(cagr.round(1).to_string())

**What it shows:** Product 3 grew from 609 claims in 2016 to 1,613 in 2018, a 165% increase in two years (CAGR +62.7%). Product 2 fell from 429 to 302, a 30% decline (CAGR -16.1%). Product 4 grew at +118.3% CAGR. Product 1 held flat (+2.3% CAGR) but peaked in 2017.

**Insight:** The percentage share charts tell the relative story; this chart tells the absolute one. Product 3 added over 1,000 claims in two years. Product 2 lost over 100. In absolute terms, the competitive gap is widening at an accelerating rate.

**Recommendation:** Set a concrete absolute claims target for Product 2, not just a share target. A goal of recovering 150 lost claims in the next 12 months, broken down by territory and HCP segment, gives the sales force something actionable to execute against.


### Chart B, Revenue / Spend Analysis (NEW)

The dataset includes Medicare-paid amount per line (`clm_line_cvrd_pd_amt`). Although claims volume is the primary KPI, revenue share tells a different and arguably more urgent story.


In [ ]:
rev_pct = revenue.div(revenue.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Stacked share
ax = axes[0]
x = np.arange(len(YEARS))
bottoms = np.zeros(len(YEARS))
for prod in PRODUCTS:
    vals = rev_pct[prod].values
    ax.bar(x, vals, bottom=bottoms, color=P_COLORS[prod], label=PROD_LABELS[prod], width=0.5, zorder=3)
    for j, (v, b) in enumerate(zip(vals, bottoms)):
        if v >= 6:
            ax.text(j, b + v / 2, f"{v:.1f}%", ha='center', va='center',
                    color='white', fontsize=10, fontweight='bold')
    bottoms += vals
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=12)
ax.set_xlabel('Calendar Year', fontsize=12)
ax.set_ylabel('Share of Medicare-Paid Amount (%)', fontsize=11)
ax.set_title('Revenue Share by Product\n(Medicare-Paid Amount, %)', fontsize=12, fontweight='bold', pad=12)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=9)

# Absolute revenue lines
ax = axes[1]
markers = ['o', 's', '^', 'D']
for prod, mkr in zip(PRODUCTS, markers):
    vals = revenue[prod].values
    ax.plot(x, vals, color=P_COLORS[prod], marker=mkr, linewidth=2.5, markersize=9, label=PROD_SHORT[prod])
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in YEARS], fontsize=12)
ax.set_xlabel('Calendar Year', fontsize=12)
ax.set_ylabel('Medicare-Paid Amount ($)', fontsize=11)
ax.set_title('Absolute Medicare Revenue by Product\n2016-2018', fontsize=12, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax.legend(loc='upper left', fontsize=9)

fig.text(0.5, 0.01,
         'J3010 surpasses J1885 in revenue share by 2018 (44.4% vs 41.6%), driven by higher avg paid per administration  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.07, right=0.97, top=0.90, bottom=0.20, wspace=0.25)
plt.show()

print("Medicare-Paid Amount by Product ($):")
print(revenue.round(0).astype(int).to_string())
print("\nRevenue Share (%):")
print(rev_pct.round(1).to_string())
print("\nAvg Paid per Administration ($):")
avg_paid = market_df.groupby(['claim_year','clm_line_hcpcs_cd'])['clm_line_cvrd_pd_amt'].mean().unstack().round(2)
print(avg_paid.to_string())

**What it shows:** Product 1's revenue share fell from 60.4% (2016) to 41.6% (2018). Product 3's revenue share rose from 30.1% to 44.4%, and now exceeds Product 1 in dollar terms even though Product 1 still leads in claims volume. Average paid per administration rose for every brand: J2250 $9.89 → $41.68; J3010 $37.08 → $63.26.

**Insight:** The financial center of gravity has already shifted to Product 3 even though the claims-share story still describes Product 1 as the leader. This raises the urgency of the case dramatically, by every measure that matters to the P&L, Product 3 is now the leader.

**Recommendation:** Reframe the leadership narrative, "we are losing the revenue race, not just the share race." Use this in the panel to ground the financial case for increased investment in Product 2.


### Chart C, HCP Writer Segmentation: J2250 vs J3010


In [ ]:
SEGS = ['Disease Aware (1)', 'Trialists (2-4)', 'Rising Stars (5-9)', 'High-Volume (10+)']
SEG_COLORS = [B5, B4, B3, B1]
seg_data = {}
for prod in ['J2250', 'J3010']:
    prod_df    = market_df[market_df['clm_line_hcpcs_cd'] == prod]
    hcp_vol    = prod_df.groupby('fac_prvdr_npi_num')['cur_clm_uniq_id'].count()
    seg_counts = pd.cut(hcp_vol, bins=[0, 1, 4, 9, 9999],
                        labels=SEGS, include_lowest=True).value_counts().reindex(SEGS, fill_value=0)
    seg_data[prod] = seg_counts

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.patch.set_facecolor('white')
fig.text(0.5, 0.97, 'HCP Writer Segmentation: Product 2 vs. Product 3  |  All Years Combined (2016-2018)',
         ha='center', va='top', fontsize=13, fontweight='bold', color='#111111')
for ax, prod, tcol in zip(axes, ['J2250', 'J3010'], [P_COLORS['J2250'], P_COLORS['J3010']]):
    vals  = seg_data[prod].values
    total = vals.sum()
    y_pos = np.arange(len(SEGS))
    ax.barh(y_pos, vals, color=SEG_COLORS, height=0.55, zorder=3)
    for i, v in enumerate(vals):
        ax.text(v + max(vals) * 0.02, i, f"{v:,}  ({v/total*100:.1f}%)",
                ha='left', va='center', fontsize=10, fontweight='bold', color=SEG_COLORS[i])
    ax.set_xlim(0, max(vals) * 1.55)
    ax.set_yticks(y_pos); ax.set_yticklabels(SEGS, fontsize=11)
    ax.set_xlabel('Number of HCPs (NPI IDs)', fontsize=11, labelpad=8)
    ax.set_title(f"{PROD_LABELS[prod]}\n(Total Writers: {total:,})",
                 fontsize=11, fontweight='bold', pad=8, color=tcol)
fig.text(0.5, 0.01,
         'Segmentation based on total unique claims per NPI across all years  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.18, right=0.97, top=0.82, bottom=0.18, wspace=0.12)
plt.show()
print("J2250 segmentation:"); print(seg_data['J2250'].to_string())
print("\nJ3010 segmentation:"); print(seg_data['J3010'].to_string())

**What it shows:** Product 2 has zero High-Volume writers (10+ claims) and only 43 Rising Stars (5-9 claims). 90% of its writers are Disease Aware or Trialists. Product 3 has 96 High-Volume writers (19.4%) and 262 Rising Stars (52.9%). Only 3.6% of Product 3 writers are Disease Aware.

**Insight:** This is the single most important chart in the entire analysis. It explains everything. Product 3 wins because its writers are deeply committed, most are high-productivity Rising Stars or High-Volume prescribers. Product 2 loses because its writers barely engage with the brand.

**Recommendation (sized prize):** Convert 20% of J2250's 284 Trialists (~57 HCPs) into Rising Stars by doubling their annual claim volume. **Result: ~+170 incremental claims/year with zero new-writer acquisition cost.** This is the highest-leverage commercial move available.


### Chart D, Writer Retention Rate by Product


In [ ]:
ret_data = {}
for prod in PRODUCTS:
    prod_df = market_df[market_df['clm_line_hcpcs_cd'] == prod]
    w = {yr: set(prod_df[prod_df['claim_year'] == yr]['fac_prvdr_npi_num']) for yr in YEARS}
    ret_data[prod] = {
        '2016→2017': len(w[2016] & w[2017]) / len(w[2016]) * 100 if w[2016] else 0,
        '2017→2018': len(w[2017] & w[2018]) / len(w[2017]) * 100 if w[2017] else 0,
    }

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
x = np.arange(len(PERIODS))
markers = ['o', 's', '^', 'D']
for prod, mkr in zip(PRODUCTS, markers):
    vals = [ret_data[prod][p] for p in PERIODS]
    ax.plot(x, vals, color=P_COLORS[prod], marker=mkr, linewidth=2.5, markersize=9, label=PROD_LABELS[prod])
    for j, v in enumerate(vals):
        ax.text(j, v + 4, f"{v:.1f}%", ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=P_COLORS[prod])
ax.set_xticks(x); ax.set_xticklabels(PERIODS, fontsize=13)
ax.set_ylim(0, 125)
ax.set_xlabel('Period', fontsize=13, labelpad=10)
ax.set_ylabel('Writer Retention Rate (%)', fontsize=12, labelpad=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('HCP Writer Retention Rate by Product\n(% of Prior-Year Writers Who Continued Writing)',
             fontsize=14, fontweight='bold', pad=16)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=9.5)
fig.text(0.5, 0.01, 'Retention Rate = Writers in both years / Writers in prior year × 100  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.1, right=0.97, top=0.88, bottom=0.28)
plt.show()
print(pd.DataFrame(ret_data).T.round(1).to_string())

**What it shows:** Product 1 retains 99%+ of its writers every year. Product 3's retention rate improved from 86.3% to 94.5%. Product 2's retention rate fell from 57.9% to 46.4%. Product 4's rate improved from 48.6% to 55.1%.

**Insight:** Less than half of Product 2's writers from 2017 continued writing in 2018. This means the brand loses more than half its writer base every year, an unsustainable commercial trajectory. Product 3 at 94.5% retention is building a compounding loyalty base. Product 2 at 46.4% is rebuilding from scratch every year.

**Recommendation:** A retention rate target of 70% for Product 2 should be set as a non-negotiable commercial objective for the next fiscal year.


### Chart E, Patient Overlap: J2250 vs J3010


In [ ]:
p2_pts  = set(market_df[market_df['clm_line_hcpcs_cd'] == 'J2250']['bene_mbi_id'])
p3_pts  = set(market_df[market_df['clm_line_hcpcs_cd'] == 'J3010']['bene_mbi_id'])
p2_only = len(p2_pts - p3_pts)
p3_only = len(p3_pts - p2_pts)
overlap = len(p2_pts & p3_pts)
p2_tot  = p2_only + overlap
p3_tot  = p3_only + overlap

fig, (ax_bar, ax_tbl) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.2, 1]})
fig.patch.set_facecolor('white')
ax_bar.bar([0], [p2_only], color=B5, width=0.5, zorder=3)
ax_bar.bar([0], [overlap], color=B2, width=0.5, bottom=p2_only, zorder=3)
ax_bar.bar([1], [p3_only], color=B4, width=0.5, zorder=3)
ax_bar.bar([1], [overlap], color=B2, width=0.5, bottom=p3_only, zorder=3)
ax_bar.text(0, p2_only/2,         f"P2 Only\n{p2_only:,}",  ha='center', va='center', fontsize=10, fontweight='bold', color=B1)
ax_bar.text(0, p2_only+overlap/2, f"Shared\n{overlap:,}",   ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax_bar.text(1, p3_only/2,         f"P3 Only\n{p3_only:,}",  ha='center', va='center', fontsize=10, fontweight='bold', color=B1)
ax_bar.text(1, p3_only+overlap/2, f"Shared\n{overlap:,}",   ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax_bar.set_xticks([0, 1]); ax_bar.set_xticklabels(['Product 2\n(J2250)', 'Product 3\n(J3010)'], fontsize=12)
ax_bar.set_ylabel('Number of Unique Patients', fontsize=12, labelpad=10)
ax_bar.set_title('Patient Overlap Between\nProduct 2 and Product 3', fontsize=13, fontweight='bold', pad=14)
ax_bar.set_ylim(0, max(p2_tot, p3_tot) * 1.25)

ax_tbl.axis('off')
tbl_data = [
    ['Product 2 Total Patients', f"{p2_tot:,}"],
    ['Product 3 Total Patients', f"{p3_tot:,}"],
    ['Shared (on Both)',         f"{overlap:,}  ({overlap/p2_tot*100:.1f}% of P2)"],
    ['Product 2 Exclusive',      f"{p2_only:,}  ({p2_only/p2_tot*100:.1f}% of P2)"],
    ['Product 3 Exclusive',      f"{p3_only:,}  ({p3_only/p3_tot*100:.1f}% of P3)"],
]
tbl = ax_tbl.table(cellText=tbl_data, colLabels=['Metric', 'Value'],
                   loc='center', cellLoc='left', bbox=[0.0, 0.1, 1.0, 0.8])
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for c in range(2):
    tbl[(0, c)].set_facecolor(B1); tbl[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, 6):
    for c in range(2):
        tbl[(r, c)].set_edgecolor('#dddddd')
        tbl[(r, c)].set_facecolor('#fff3cd' if r == 3 else (B6 if r % 2 == 0 else 'white'))
        if r == 3: tbl[(r, c)].set_text_props(fontweight='bold', color='#856404')
ax_tbl.set_title('Patient Pool Summary', fontsize=11, fontweight='bold', pad=8, color=B1)
fig.suptitle('Patient Overlap Analysis: Product 2 vs. Product 3  |  2016-2018', fontsize=14, fontweight='bold', y=1.02)
fig.text(0.5, 0.01,
         '58% of Product 2 patients have also received Product 3, direct evidence of competitive displacement  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.12, wspace=0.15)
plt.show()
print(f"P2 patients: {p2_tot:,} | P3 patients: {p3_tot:,} | Shared: {overlap:,} ({overlap/p2_tot*100:.1f}% of P2)")

**What it shows:** 570 out of 982 Product 2 patients, 58%, have also received Product 3 at some point. Only 412 patients are exclusively on Product 2. Product 3 has 1,509 patients who have never received Product 2.

**Insight:** The majority of Product 2's patient base is not loyal, they are already being treated with the competitor as well. Product 3 is not just winning new patients, it is treating *our* patients. This is the clearest possible evidence of active competitive displacement at the patient level.

**Recommendation:** Identify the HCPs treating the 570 shared patients. These providers are currently using both products for similar patients, meaning they have not made a definitive clinical preference yet. A targeted clinical differentiation campaign aimed specifically at these HCPs is the highest single conversion lever in the dataset.


### Chart F, Patient Switcher Matrix: 2016 → 2018 (NEW)

A direct test of the cannibalization claim: of patients on each brand in 2016, where were they in 2018?


In [ ]:
pf = market_df[['bene_mbi_id', 'claim_year', 'clm_line_hcpcs_cd']].drop_duplicates()
yr16 = pf[pf['claim_year'] == 2016].groupby('bene_mbi_id')['clm_line_hcpcs_cd'].apply(set)
yr18 = pf[pf['claim_year'] == 2018].groupby('bene_mbi_id')['clm_line_hcpcs_cd'].apply(set)

rows = []
for prod_2016 in PRODUCTS:
    cohort = yr16[yr16.apply(lambda s: prod_2016 in s)].index
    total = len(cohort)
    if total == 0: continue
    churned = sum(1 for p in cohort if p not in yr18.index)
    moves = {q: 0 for q in PRODUCTS}
    for p in cohort:
        if p in yr18.index:
            for q in PRODUCTS:
                if q in yr18[p]:
                    moves[q] += 1
    rows.append({
        'Origin (2016)': prod_2016,
        'Cohort': total,
        'Lapsed (no 2018 claims)': churned,
        **{f'On {q} 2018': moves[q] for q in PRODUCTS},
    })
matrix_df = pd.DataFrame(rows).set_index('Origin (2016)')
print("Patient cohort transition matrix (2016 → 2018):")
print(matrix_df.to_string())

# Visualize as a heatmap-style table
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor('white')
ax.axis('off')
cols     = ['Cohort', 'Lapsed (no 2018)', 'On J1885 2018', 'On J2250 2018', 'On J2704 2018', 'On J3010 2018']
cell_text = []
for prod in PRODUCTS:
    row = matrix_df.loc[prod]
    cell_text.append([
        f"{int(row['Cohort']):,}",
        f"{int(row['Lapsed (no 2018 claims)']):,} ({row['Lapsed (no 2018 claims)']/row['Cohort']*100:.0f}%)",
        f"{int(row['On J1885 2018']):,} ({row['On J1885 2018']/row['Cohort']*100:.0f}%)",
        f"{int(row['On J2250 2018']):,} ({row['On J2250 2018']/row['Cohort']*100:.0f}%)",
        f"{int(row['On J2704 2018']):,} ({row['On J2704 2018']/row['Cohort']*100:.0f}%)",
        f"{int(row['On J3010 2018']):,} ({row['On J3010 2018']/row['Cohort']*100:.0f}%)",
    ])
row_labels = [PROD_SHORT[p] for p in PRODUCTS]
tbl = ax.table(cellText=cell_text, rowLabels=row_labels, colLabels=cols,
               loc='center', cellLoc='center', bbox=[0.0, 0.0, 1.0, 1.0])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for c in range(len(cols)):
    tbl[(0, c)].set_facecolor(B1); tbl[(0, c)].set_text_props(color='white', fontweight='bold')
for r in range(1, len(PRODUCTS) + 1):
    for c in range(len(cols)):
        tbl[(r, c)].set_edgecolor('#dddddd')
        tbl[(r, c)].set_facecolor(B6 if r % 2 == 0 else 'white')
    tbl[(r, -1)].set_text_props(color=list(P_COLORS.values())[r - 1], fontweight='bold')
ax.set_title('Patient-Level Switcher Matrix: 2016 Cohort → Brand of Record in 2018',
             fontsize=12, fontweight='bold', pad=10, color=B1)
fig.text(0.5, 0.01,
         "Read each row: 'Of patients on this brand in 2016, here is where they were in 2018'. Cohorts may overlap.  |  Source: Medicare CCLF Claims",
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.10, right=0.98, top=0.86, bottom=0.10)
plt.show()

**What it shows:** Of the 2,017 patients on Product 1 in 2016, by 2018 only ~6% had been on Product 2, but ~34% had been on Product 3. The intended portfolio handover from J1885 to J2250 did not occur in any meaningful volume.

**Insight:** This is the textbook cannibalization-failure metric and it directly answers the case question. Product 3 is the destination brand for Product 1 patients, not Product 2. The portfolio strategy is failing at the patient-flow level, not just the share level.

**Recommendation:** Use this matrix in the panel presentation as the proof-point that the cannibalization plan needs to be replaced, not refined. The brand strategy must shift from "absorb Product 1 patients" to "compete for the induction/sedation segment against Product 4 and Product 3.


### Chart G, Quarterly Trend (NEW)

Identifying inflection points within the three-year window.


In [ ]:
q_pivot = market_df.groupby(['claim_quarter', 'clm_line_hcpcs_cd']).size().unstack(fill_value=0).reindex(columns=PRODUCTS, fill_value=0)
q_pivot = q_pivot.sort_index()

fig, ax = plt.subplots(figsize=(13, 5.5))
fig.patch.set_facecolor('white')
x = np.arange(len(q_pivot.index))
markers = ['o', 's', '^', 'D']
for prod, mkr in zip(PRODUCTS, markers):
    ax.plot(x, q_pivot[prod].values, color=P_COLORS[prod], marker=mkr, linewidth=2,
            markersize=7, label=PROD_LABELS[prod])
ax.set_xticks(x); ax.set_xticklabels(q_pivot.index, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Calendar Quarter', fontsize=12, labelpad=10)
ax.set_ylabel('Claims Volume', fontsize=12, labelpad=10)
ax.set_title('Quarterly Claims Trend by Product\nDetecting Inflection Points in Brand Trajectory',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, axis='y', linestyle='--', linewidth=0.5, color='#dddddd')
fig.text(0.5, 0.01,
         'J3010 inflection between Q3 2017 and Q1 2018, sharpest acceleration period  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.22)
plt.show()
print("Quarterly claims by product:")
print(q_pivot.to_string())

**What it shows:** J3010 momentum accelerates between Q3 2017 and Q1 2018, the most pronounced acceleration in any brand across the entire window. J2250's decline becomes consistent from Q4 2017 onward.

**Insight:** Q3 2017 to Q1 2018 is the most likely competitive event window, formulary win, sales-force expansion, KOL launch, or sample-program ramp on the competitor side. We do not have CRM/calls data to confirm, but this is the time period that field intel and competitive intelligence teams should investigate.

**Recommendation:** Brief the field team to investigate any institutional decisions (formulary wins, hospital protocol updates) made during Q3 2017, Q1 2018 in the 5 emergency territories. The trigger for J3010's inflection happened in this window.


### Chart H, Full Territory Ranking for J2250 (YoY 2017→2018)


In [ ]:
terr_t = market_df[market_df['clm_line_hcpcs_cd'] == 'J2250'].groupby(
    ['Territory Name', 'claim_year'])['cur_clm_uniq_id'].count().unstack(fill_value=0)
terr_t.columns = [int(c) for c in terr_t.columns]
for y in YEARS:
    if y not in terr_t.columns: terr_t[y] = 0
terr_t = terr_t[YEARS].copy()
terr_t['yoy_1718'] = (terr_t[2018] - terr_t[2017]) / terr_t[2017].replace(0, np.nan) * 100
terr_t = terr_t.sort_values('yoy_1718')

def bar_color(v):
    if v <= -50: return '#1a3a5c'
    elif v <= -25: return '#2e86c1'
    elif v < 0: return '#7fb3d3'
    else: return '#aed6f1'

colors = [bar_color(v) for v in terr_t['yoy_1718'].values]
y_pos  = np.arange(len(terr_t))

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor('white')
ax.barh(y_pos, terr_t['yoy_1718'].values, color=colors, height=0.65, zorder=3)
for i, v in enumerate(terr_t['yoy_1718'].values):
    offset = -1.5 if v < 0 else 1.5
    ax.text(v + offset, i, f"{v:+.1f}%", ha='right' if v < 0 else 'left', va='center',
            fontsize=9, fontweight='bold', color=bar_color(v))
ax.axvline(0, color='#333333', linewidth=1.2, zorder=4)
ax.set_yticks(y_pos); ax.set_yticklabels(terr_t.index, fontsize=10)
ax.set_xlabel('YoY Change in Claims Volume (%)', fontsize=12, labelpad=10)
ax.set_title('Product 2 (Variant, J2250): Year-over-Year Claims Change by Territory\n2017 → 2018  |  All 22 Territories Ranked',
             fontsize=13, fontweight='bold', pad=16)
legend_elements = [
    mpatches.Patch(facecolor='#1a3a5c', label='Critical Decline (≤ -50%)'),
    mpatches.Patch(facecolor='#2e86c1', label='Severe Decline (-25% to -50%)'),
    mpatches.Patch(facecolor='#7fb3d3', label='Moderate Decline (0% to -25%)'),
    mpatches.Patch(facecolor='#aed6f1', label='Stable / Growing (> 0%)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
fig.text(0.5, 0.01, '15 of 22 territories declined for Product 2 from 2017 to 2018  |  Source: Medicare CCLF Claims',
         ha='center', fontsize=8.5, color='#888888', style='italic')
plt.subplots_adjust(left=0.22, right=0.97, top=0.92, bottom=0.08)
plt.show()
print(terr_t[['yoy_1718']].round(1).to_string())

**What it shows:** 15 of 22 territories declined for Product 2 from 2017 to 2018. Four territories are in critical decline (worse than -50%): St. Louis, Phoenix, LA-San Diego, New York. Six territories are stable or growing: Pittsburg (+30%), Detroit (+20%), Charlotte (+20%), Houston (+14.3%), Chicago (+11.1%), San Jose (+9.5%).

**Insight:** The decline is broad and geographic, 68% of territories are contracting. But 6 territories are growing, and these are the commercial benchmarks. Understanding what the sales force is doing differently in Pittsburg, Detroit, and Charlotte could provide a replicable playbook.

**Recommendation:** Conduct a commercial-excellence review of the 6 growing territories. Interview the reps, analyze their call patterns, review their HCP target lists. Extract the specific behaviors and tactics driving growth and package them into a standardized playbook. Deploy that playbook in the 4 emergency territories.


---

# Q3, Strategy to Stop Market Share Erosion and Gain Traction

The analysis across Q1, Q2, and Additional Strategic Analytics points to four non-negotiable priorities. Each carries a sized prize, an effort estimate, a 30/60/90 day plan, and a tracked KPI.

---

### Priority 1, Fix Writer Retention Before Spending on Acquisition

**Problem:** 46.4% retention means we lose more than half of our writers every year. No acquisition program survives this churn rate.

**Action:** Build the NPI list of J2250 writers active in 2017 but absent in 2018 (~148 NPIs). Personally route each to the most senior territory rep. Run a structured re-engagement protocol, clinical case review, peer-to-peer programs, samples, patient-support co-promotion.

**Sized prize:** Lift retention from 46.4% → 65% over 12 months ≈ recover ~50-90 writers ≈ **+120 to +200 incremental claims** (assuming 2.5 claims/writer baseline).

**30/60/90:** 30 days, extract NPI list, segment by territory, brief reps. 60 days, first-touch outreach to 100% of list. 90 days, track re-engagement conversion, refine playbook.

**KPI:** Monthly writer retention rate vs prior cohort.

---

### Priority 2, Convert Trialists into Rising Stars

**Problem:** J2250 has zero High-Volume and only 43 Rising Star writers. 90% of its base sits at 1-4 claims/year.

**Action:** Targeted Trialist program, quarterly clinical case studies, dosing-flexibility detail aids for elderly patients (the dominant 61+ segment), peer speaker events featuring our highest-volume J2250 writer.

**Sized prize:** Convert 20% of 284 Trialists (~57 HCPs) from 3 → 6 claims/year ≈ **+170 incremental claims/year**, zero acquisition cost. At observed 2018 avg paid/line of $41.68, ~$7K direct Medicare revenue uplift, with significant multiplier from commercial book.

**30/60/90:** 30 days, define Trialist target list. 60 days, deploy detail aids and case studies. 90 days, measure claims-per-HCP shift.

**KPI:** % of Trialists migrating to Rising Star tier per quarter.

---

### Priority 3, Declare Four Emergency Territories

**Problem:** St. Louis (-72.7%), Phoenix (-70.0%), LA-San Diego (-57.9%), New York (-57.5%) lost over half their J2250 volume in 2018. J3010 added 165+ claims in those same four markets.

**Action:** Audit rep activity logs and formulary status in each territory within 30 days. For NY and LA, increase rep call frequency immediately and run NPP campaigns. For St. Louis and Phoenix, small-market focused programs.

**Sized prize:** Recover even 50% of 2018 lost volume in these 4 markets = **~+30 claims** in the short term, plus protection of the writer base for compounding gains.

**30/60/90:** 30 days, root-cause audit (rep, formulary, KOL). 60 days, corrective interventions launched. 90 days, review trajectory; escalate to the next 5 declining territories if needed.

**KPI:** Monthly claims volume per territory vs Q4 2018 baseline.

---

### Priority 4, Defend the Anesthesiology Base

**Problem:** Anesthesiology is 47% of all market writers (228 NPIs). It is the must-win specialty, and it is exactly where J3010 is consolidating.

**Action:** Named-NPI ring-fence around the top 30 anesthesiology writers in our base. KOL programs, exclusive clinical content, advisory boards, real-world evidence partnerships.

**Sized prize:** Hold this segment at 2017 share = avoid losing the next ~150 claims per year that would otherwise leak to J3010.

**KPI:** Claim share within Anesthesiology specialty by quarter.

---

### What we will *not* pursue (discipline list)

- **Net-new HCP recruitment** until retention is above 65%. Acquiring writers we cannot retain is wasted spend.
- **National brand campaigns.** The data is geographic and HCP-specific, broadcast spend dilutes ROI.
- **Repositioning vs J1885.** Section 3 (pharmacological reframe) makes this competitive frame obsolete. We compete with J2704 and J3010, not J1885.


---

# Q4, Data Exploration Opportunities (Qualitative)

The Medicare CCLF claims dataset answers *what* happened. To answer *why* and *what to do next*, the following datasets are required.

| Dataset | What it would unlock | Decision it would change |
| --- | --- | --- |
| **Xponent (IQVIA)** | HCP-level prescription data across all payers; NRx and TRx trends. Identify exactly which HCPs switched from J2250 → J3010 and when. | Replaces our inferred switcher matrix with ground truth and lets us name the specific NPIs to re-engage. |
| **DDD (Direct to Distributor)** | Hospital-level and pharmacy-level brand purchasing. | Reveals whether the loss is a formulary-access issue (institution-level) or a detail/HCP issue. Different intervention. |
| **NPA / NSP (National Prescription Audit)** | True total-market share across commercial + Medicaid + Medicare. | Confirms whether Medicare data understates or overstates our share. May redirect priorities to commercial book. |
| **CRM / Veeva Call Activity** | Calls per HCP, message recall, sample drops, NPP delivery. | Lets us calculate calls-to-claims conversion and isolate whether decline is a coverage or message problem. |
| **Speaker programs & samples response data** | Attendance, response, sample-to-claim conversion. | Allows replication of J3010's likely education push from 2016 (when they added 344 new writers). |
| **Payer / formulary coverage** | Tier placement, prior-auth requirements per payer. | One regional formulary win can explain an entire territory's decline, currently invisible. |
| **IQVIA Channel Dynamics** | NRx vs TRx by channel (retail vs hospital vs LTC). | Shows whether the variant brand is failing at acquisition or refill stages. |
| **Adherence / persistency (PDC, MPR)** | Days-on-therapy, refill behaviour. | Distinguishes patient-level abandonment from HCP switching. |
| **Primary research / VOC** | Message recall, attribute trade-off, satisfaction scores. | Tells us why HCPs are choosing J3010 over J2250, without this we are guessing. |
| **Competitive intelligence** | J3010 launch tactics, pricing, sales-force size, sample drops. | Currently inferred indirectly; needed for replication. |

---

## Data Limitations

- **Medicare-only.** Conclusions cannot be safely generalized to commercial or Medicaid books without NPA/Xponent.
- **Sample size in tail territories.** Several "biggest decline" territories have <30 claims/year, large percentage swings can be partly noise. Recommendations have been kept directional and use absolute claim counts where possible.
- **Boundary effect for "new writers".** 2016 counts equal total writers because the data window starts in 2016. True acquisition trend is in 2017 and 2018 only.
- **Diagnosis specialty is line-level, not principal.** Checked as supplementary; conclusions unchanged.
- **Upstream typos in territory data** (e.g., *Philedelphia*, *Pittsburg*) preserved as-is; no impact on numerical conclusions.
- **No payer mix, calls, samples, NRx, or formulary data.** See Q4 above, these are the highest-priority gaps to close.

---

*End of submission. Analysis derived solely from Medicare CCLF claims and accompanying mapping files. Recommendations are directional and reflect the constraints noted above.*
